# Bell State — Qiskit Implementation

**Primitive:** Bell state preparation, |Phi+> = (|00> + |11>) / sqrt(2)

**SETUP (assumed defaults)**
- N QUBITS: 2
- GATES (shared source of truth, used for circuit build AND diagram):
  1. H on qubit 0
  2. CX, control = qubit 0, target = qubit 1
- QUBIT ORDERING CONTRACT: qubit 0 = rightmost bit in every printed bitstring
- SHOTS: 10000, SEED: 42
- GLOBAL PHASE: normalized so first nonzero amplitude (ascending bitstring order) is real and positive

**Theory recap:** A Bell state is the maximally entangled two-qubit state
|Phi+> = (|00> + |11>) / sqrt(2). It is prepared by putting qubit 0 into an
equal superposition with a Hadamard gate, then using qubit 0 as the control
of a CNOT (CX) targeting qubit 1. The CNOT correlates the two qubits so that
measuring one qubit fully determines the other: whenever qubit 0 flips to
|1>, qubit 1 is forced to flip along with it, entangling the pair.

H = 1/sqrt(2) * [[1, 1], [1, -1]]

CX (control=q0, target=q1): flips q1 iff q0 = 1.

**Bit-ordering note:** this notebook does not assume qubit-0-rightmost is
Qiskit's native convention — it verifies it empirically in Section 1 with a
calibration circuit before trusting any downstream result.

**IMPORTANT: Run every cell in order, top to bottom** (Kernel -> Restart & Run All).

In [ ]:
# Run this cell first if you get ModuleNotFoundError or MissingOptionalLibraryError.
# pylatexenc is required for circuit.draw(output="mpl") gate-label rendering.
!pip install -q qiskit qiskit-aer pylatexenc
# Restart the kernel after this finishes (Kernel -> Restart Kernel), then Run All.


In [ ]:
# --- Imports ---
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

np.random.seed(42)

# --- SETUP constants ---
N_QUBITS = 2
SHOTS = 10000
SEED = 42

# GATES: shared source of truth for circuit build AND circuit diagram.
# Format: (gate_name, qubits_list, param). For "h": qubits=[target]. For "cx": qubits=[control, target].
GATES = [
    ("h",  [0],    None),
    ("cx", [0, 1], None),
]
print("Gate sequence:", GATES)

## 1. Bit-ordering calibration (verify, don't assume)

We apply X to the *highest-index* qubit only (qubit 1) and check which native statevector index lights up. Under the qubit-0-rightmost CONTRACT, the expected bitstring is `"10"` (leftmost char = qubit 1 = 1, rightmost char = qubit 0 = 0). If Qiskit's native output already reads this way, no reversal is needed; if it reads reversed, `NEEDS_REVERSAL` is set True and used everywhere downstream instead of a hardcoded assumption.

In [ ]:
calib_qc = QuantumCircuit(N_QUBITS)
calib_qc.x(N_QUBITS - 1)  # flip qubit 1 only, leave qubit 0 at |0>
calib_sv = Statevector.from_instruction(calib_qc)

nonzero_idx = int(np.argmax(np.abs(calib_sv.data)))
native_bitstring = format(nonzero_idx, f"0{N_QUBITS}b")
expected_contract_bitstring = "1" + "0" * (N_QUBITS - 1)  # qubit (n-1) = leftmost = 1, rest 0

if native_bitstring == expected_contract_bitstring:
    NEEDS_REVERSAL = False
elif native_bitstring[::-1] == expected_contract_bitstring:
    NEEDS_REVERSAL = True
else:
    raise RuntimeError(f"Calibration inconclusive: got '{native_bitstring}', "
                        f"expected '{expected_contract_bitstring}' or its reverse.")

print(f"Native bitstring for X(qubit {N_QUBITS-1}) only: '{native_bitstring}'")
print(f"Expected under contract (qubit0=rightmost):        '{expected_contract_bitstring}'")
print(f"NEEDS_REVERSAL = {NEEDS_REVERSAL}")

## 2. Build the circuit explicitly from the shared GATES list

In [ ]:
qc = QuantumCircuit(N_QUBITS, N_QUBITS, name="bell_state")

for gate_name, qubits, param in GATES:
    if gate_name == "h":
        qc.h(qubits[0])
    elif gate_name == "cx":
        qc.cx(qubits[0], qubits[1])
    else:
        raise ValueError(f"Unhandled gate in GATES: {gate_name}")

qc.barrier()
qc.draw(output="mpl")

## 3. Draw the circuit and save to file

In [ ]:
fig = qc.draw(output="mpl")
fig.savefig("bell_state_qiskit_circuit.png", bbox_inches="tight")
fig

## 4. Explicit matrix representation of the key operator(s)

In [ ]:
H_matrix = (1 / np.sqrt(2)) * np.array([[1, 1],
                                          [1, -1]], dtype=complex)
print("H matrix:")
print(np.round(H_matrix, 4))
print()

# CX with control=qubit0, target=qubit1, in basis {|00>,|01>,|10>,|11>} (qubit0 = rightmost)
CX_matrix = np.zeros((4, 4), dtype=complex)
for idx in range(4):
    q1, q0 = idx // 2, idx % 2         # idx = 2*q1 + q0  (qubit0 = LSB = rightmost)
    new_q1 = (1 - q1) if q0 == 1 else q1   # flip q1 iff control q0 == 1
    new_idx = 2 * new_q1 + q0
    CX_matrix[new_idx, idx] = 1
print("CX (control=q0, target=q1) matrix, basis {|00>,|01>,|10>,|11>}:")
print(np.round(CX_matrix.real, 4))

In [ ]:
prep_circuit = QuantumCircuit(N_QUBITS, name="prep_only")
for gate_name, qubits, param in GATES:
    if gate_name == "h":
        prep_circuit.h(qubits[0])
    elif gate_name == "cx":
        prep_circuit.cx(qubits[0], qubits[1])

full_op = Operator(prep_circuit)
print(f"Full {2**N_QUBITS}x{2**N_QUBITS} operator (basis {{|00>, |01>, |10>, |11>}}):")
print(np.round(full_op.data, 4))

# Cross-check: full_op should equal CX_matrix @ (I kron H)
I2 = np.eye(2, dtype=complex)
manual_full_op = CX_matrix @ np.kron(I2, H_matrix)
assert np.allclose(full_op.data, manual_full_op, atol=1e-6), "Operator mismatch between Qiskit and manual construction!"
print()
print("Cross-check against manually constructed CX @ (I kron H): MATCH")

## 5. Exact statevector (before measurement), reordered if needed, global-phase normalized

In [ ]:
sv = Statevector.from_instruction(qc)
amps = sv.data.copy()

def reorder_if_needed(amps, n_qubits, needs_reversal):
    if not needs_reversal:
        return amps.copy()
    dim = 2 ** n_qubits
    reordered = np.zeros(dim, dtype=complex)
    for native_idx in range(dim):
        native_bits = format(native_idx, f"0{n_qubits}b")
        target_bits = native_bits[::-1]
        target_idx = int(target_bits, 2)
        reordered[target_idx] = amps[native_idx]
    return reordered

def normalize_global_phase(statevector_data, n_qubits):
    dim = 2 ** n_qubits
    bitstrings = [format(i, f"0{n_qubits}b") for i in range(dim)]
    order = np.argsort(bitstrings)
    for idx in order:
        amp = statevector_data[idx]
        if abs(amp) > 1e-9:
            phase = np.angle(amp)
            corrected = statevector_data * np.exp(-1j * phase)
            return corrected, bitstrings
    return statevector_data, bitstrings

amps_contract = reorder_if_needed(amps, N_QUBITS, NEEDS_REVERSAL)
amps_norm, bitstrings = normalize_global_phase(amps_contract, N_QUBITS)

exact_probs = {}
for i, bs in enumerate(bitstrings):
    exact_probs[bs] = round(abs(amps_norm[i]) ** 2, 4)

exact_probs = dict(sorted(exact_probs.items()))
print("Exact statevector amplitudes (contract order, phase-normalized):")
print(np.round(amps_norm, 4))
print()
print("Exact statevector probabilities (qubit0 = rightmost):")
print(exact_probs)

## 6. Measurement: add measurements, run AerSimulator, plot histogram

In [ ]:
qc_measured = qc.copy()
qc_measured.measure(range(N_QUBITS), range(N_QUBITS))

sim = AerSimulator(seed_simulator=SEED)
job = sim.run(qc_measured, shots=SHOTS, seed_simulator=SEED)
result = job.result()
raw_counts = result.get_counts()

def reorder_counts_if_needed(counts, needs_reversal):
    if not needs_reversal:
        return dict(counts)
    out = {}
    for bs, c in counts.items():
        rbs = bs[::-1]
        out[rbs] = out.get(rbs, 0) + c
    return out

counts = reorder_counts_if_needed(raw_counts, NEEDS_REVERSAL)

fig2 = plot_histogram(counts)
fig2.savefig("bell_state_qiskit_histogram.png", bbox_inches="tight")
fig2

In [ ]:
sampled_probs = {bs: round(c / SHOTS, 4) for bs, c in counts.items()}
sampled_probs = dict(sorted(sampled_probs.items()))
print(f"Sampled probabilities ({SHOTS} shots, seed={SEED}):")
print(sampled_probs)

## 7. Circuit resource counts

In [ ]:
print(f"Depth: {qc_measured.depth()}")
print(f"Gate/operation count: {qc_measured.size()}")
print(f"Gates in shared GATES spec: {len(GATES)}  (h + cx, matches circuit build)")

## 8. Convention notes

In [ ]:
print(f"Bit ordering: calibration in Section 1 found NEEDS_REVERSAL = {NEEDS_REVERSAL}.")
print("For Qiskit, Statevector/get_counts() natively index qubit 0 as the rightmost")
print("(least-significant) bit, which already matches the qubit0=rightmost CONTRACT --")
print("so no reversal was needed, confirmed empirically rather than assumed.")
print()
print("Global phase: normalized by finding the first nonzero amplitude in ascending-bitstring")
print("order and rotating the whole statevector so that amplitude becomes real and positive.")
print("For this primitive, both nonzero amplitudes (|00>, |11>) are real and equal by")
print("construction, so this normalization step is a no-op here (unlike Arbitrary States).")

## 9. Hand-derived expected result (independent sanity check)

Derived directly from the gate definitions, independent of the code above: H puts qubit 0 into (|0>+|1>)/sqrt(2); CX(control=q0, target=q1) then maps |0>_q0|0>_q1 -> |00> and |1>_q0|0>_q1 -> |1>_q0|1>_q1, giving (|00> + |11>) / sqrt(2). Expected probabilities: `{'00': 0.5, '01': 0.0, '10': 0.0, '11': 0.5}`.

In [ ]:
EXPECTED_PROBS = {"00": 0.5, "01": 0.0, "10": 0.0, "11": 0.5}

max_delta_vs_hand = max(abs(exact_probs.get(k, 0) - EXPECTED_PROBS.get(k, 0))
                         for k in set(exact_probs) | set(EXPECTED_PROBS))
print("Max delta vs. hand-derived expectation:", max_delta_vs_hand)
assert max_delta_vs_hand < 1e-6, "Computed result does not match hand-derived Bell state probabilities!"
print("Sanity check PASSED: exact_probs matches the hand-derived expectation.")

## Final result (for diffing against the CUDA-Q version)

In [ ]:
QISKIT_RESULT = exact_probs
print("QISKIT_RESULT =", QISKIT_RESULT)

## 10. Diff check against CUDA-Q

Paste the `CUDAQ_RESULT` dict printed at the end of the CUDA-Q notebook below, then run this cell.

In [ ]:
CUDAQ_RESULT = {'00': 0.5, '01': 0.0, '10': 0.0, '11': 0.5}  # <-- paste from the CUDA-Q notebook's printed output

all_keys = sorted(set(QISKIT_RESULT) | set(CUDAQ_RESULT))
max_delta = max(abs(QISKIT_RESULT.get(k, 0) - CUDAQ_RESULT.get(k, 0)) for k in all_keys)
print("Max probability delta:", max_delta)
print("Match within tolerance:", max_delta < 1e-3)
assert max_delta < 1e-3, "Qiskit and CUDA-Q results disagree beyond tolerance!"
print("PASS: Qiskit and CUDA-Q results agree.")